## 1. 환경설정

In [121]:
import openai
import openai
from openai import OpenAI
import tiktoken

In [114]:
import os
from pymongo import MongoClient
from dotenv import load_dotenv

In [122]:
import os
import time
from pymongo import MongoClient
from openai import OpenAI, APIError, RateLimitError

In [115]:
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

In [117]:
try:
    mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)

    mongo_client.admin.command('ping')
    print("✅ [성공] MongoDB 클라우드와 연결되었습니다!")
    print(f"📂 현재 존재하는 DB 목록: {mongo_client.list_database_names()}")

except Exception as e:
    print("❌ [실패] 연결에 문제가 발생했습니다.")
    print(f"👉 에러 내용: {e}") 

✅ [성공] MongoDB 클라우드와 연결되었습니다!
📂 현재 존재하는 DB 목록: ['recipe_project', 'admin', 'local']


In [118]:
db = mongo_client["recipe_project"]  # DB 이름
collection = db["recipes"]

In [119]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [120]:
if not OPENAI_API_KEY:
    print("🚨 [에러] API 키가 비어있습니다. .env 파일에 OPENAI_API_KEY가 제대로 적혀있는지 확인하세요!")
else:
    print("✅ API 키 로드 성공!")

✅ API 키 로드 성공!


In [ ]:
open_client = OpenAI(api_key=OPENAI_API_KEY)

## 2. mongodb에 ingredients의 embedding 넣었던 과정들 (실제 깃허브에는 들어갈 필요없음)

In [ ]:
cursor = collection.find({}, {"ingredients": 1})

In [8]:
processed_data = []

In [9]:
for doc in cursor:
    if "ingredients" in doc:
        # 전처리 로직
        cleaned = ", ".join([item.replace(" 구매", "").strip() for item in doc["ingredients"]])
        
        processed_data.append({
            "id": doc["_id"],
            "text_for_embedding": cleaned
        })

In [10]:
print(processed_data[0])

{'id': ObjectId('69d36d1d2e5774f9b4ca1f4f'), 'text_for_embedding': '어묵 200g, 무 150g, 대파 조금, 물 1L, 코인육수 1개, 국간장 1T, 참치액 1T, 소금 약간, 후추 톡톡'}


In [14]:
def get_ingredients_embedding(text):
    model = "text-embedding-3-small"
    
    # 토큰 수 확인
    encoding = tiktoken.get_encoding("cl100k_base")
    token_count = len(encoding.encode(text))
    
    # 임베딩 생성 요청 (openai_client 사용)
    response = open_client.embeddings.create(
        input=text,
        model=model
    )
    
    return response.data[0].embedding

In [15]:
from pymongo import UpdateOne
import time

In [16]:
batch_size = 1000
total_docs = len(processed_data)
model = "text-embedding-3-small"

In [17]:
print(f"총 {total_docs}개의 데이터를 {batch_size}개씩 나누어 임베딩 및 저장을 시작합니다.")

총 56028개의 데이터를 1000개씩 나누어 임베딩 및 저장을 시작합니다.


In [18]:
valid_processed_data = [
    item for item in processed_data 
    if item["text_for_embedding"].strip() != ""
]

In [19]:
print(f"원본 데이터: {len(processed_data)}개")
print(f"빈 데이터 제외 후: {len(valid_processed_data)}개")
print("-" * 30)

원본 데이터: 56028개
빈 데이터 제외 후: 54108개
------------------------------


In [20]:
batch_size = 1000
total_docs = len(valid_processed_data)
model = "text-embedding-3-small"

In [21]:
print(f"총 {total_docs}개의 유효한 데이터를 {batch_size}개씩 나누어 임베딩 및 저장을 시작합니다.")

총 54108개의 유효한 데이터를 1000개씩 나누어 임베딩 및 저장을 시작합니다.


In [22]:
from pymongo import UpdateOne
import time

In [23]:
# ==========================================
# 1. 환경 설정 (다이어트 & 안전 설정)
# ==========================================
batch_size = 500               # 한 번에 처리할 개수 (메모리 안정성 유지)
model = "text-embedding-3-small"
reduced_dimensions = 256       # 핵심! 벡터 크기를 1/6로 줄여서 용량 차지 최소화

# 🚨 안전장치: Atlas 무료 티어(512MB)를 지키기 위한 마지노선 설정
MAX_SAFE_STORAGE_MB = 480      # 480MB가 넘어가면 멈추도록 설정합니다.

print(f"🚀 {reduced_dimensions}차원 압축 임베딩을 시작합니다!")
print(f"방어 모드 가동: DB 용량이 {MAX_SAFE_STORAGE_MB}MB를 넘으면 자동 정지됩니다.\n")

🚀 256차원 압축 임베딩을 시작합니다!
방어 모드 가동: DB 용량이 480MB를 넘으면 자동 정지됩니다.



In [24]:
# ==========================================
# 2. 용량 확인 함수 (안전 요원 역할)
# ==========================================
def check_db_storage(db):
    """현재 데이터베이스의 실제 차지 용량(MB)을 확인합니다."""
    stats = db.command("dbstats")
    # MongoDB Atlas는 물리적 점유 공간(storageSize)을 기준으로 제한을 둡니다.
    storage_mb = stats.get("storageSize", 0) / (1024 * 1024)
    return storage_mb

In [25]:
# ==========================================
# 3. 데이터 가져오기 (스트리밍 방식)
# ==========================================
# 이미 임베딩이 있는 문서(완료된 것)는 건너뛰고, 안 된 것만 가져옵니다.
cursor = collection.find(
    {
        "ingredients": {"$exists": True}, 
        "ingredients_embedding": {"$exists": False} 
    },
    {"ingredients": 1, "_id": 1}
)

In [26]:
current_batch_texts = []
current_batch_ids = []
processed_count = 0

In [27]:
# ==========================================
# 4. 본격적인 반복 작업 시작
# ==========================================
for doc in cursor:
    # 4-1. 텍스트 전처리 (빈칸이나 '구매' 글자 등 불필요한 내용 제거)
    raw = doc.get("ingredients", [])
    cleaned = ", ".join([item.replace(" 구매", "").strip() for item in raw if item.strip()])
    
    # 전처리 후 내용이 없으면 이 레시피는 건너뜁니다.
    if not cleaned: 
        continue
        
    current_batch_texts.append(cleaned)
    current_batch_ids.append(doc["_id"])
    
    # 4-2. 바구니(batch)가 다 찼으면(500개) 처리 시작
    if len(current_batch_texts) >= batch_size:
        # 🚨 처리하기 전에 먼저 현재 용량부터 확인합니다!
        current_storage = check_db_storage(db)
        if current_storage >= MAX_SAFE_STORAGE_MB:
            print(f"\n🛑 [긴급 정지] 현재 용량이 {current_storage:.2f}MB에 도달했습니다!")
            print("DB가 터지는 것을 막기 위해 작업을 안전하게 중단합니다.")
            break # 반복문을 완전히 빠져나감
            
        try:
            # OpenAI API 호출 (반드시 dimensions 파라미터가 들어가야 용량이 줄어듭니다)
            response = open_client.embeddings.create(
                input=current_batch_texts, 
                model=model,
                dimensions=reduced_dimensions 
            )
            
            # MongoDB에 한 번에 업데이트할 보따리(ops) 만들기
            ops = [
                UpdateOne({"_id": _id}, {"$set": {"ingredients_embedding": data.embedding}})
                for _id, data in zip(current_batch_ids, response.data)
            ]
            
            # 벌크 업데이트 슛!
            collection.bulk_write(ops)
            processed_count += len(ops)
            
            print(f"✅ {processed_count}개 처리 완료... (현재 DB 용량: {current_storage:.2f}MB)")
            
            # 바구니를 비워주고 다음 작업을 준비합니다.
            current_batch_texts, current_batch_ids = [], []
            time.sleep(0.5) # API 한도 초과 방지를 위한 0.5초 휴식
            
        except Exception as e:
            print(f"\n❌ 작업 중 에러 발생: {e}")
            break # 에러가 나면 다음 작업을 무리하게 진행하지 않고 멈춤

# ==========================================
# 5. 마지막 남은 잔여 데이터(500개 미만) 처리
# ==========================================
# 위에서 용량 초과로 브레이크(break)가 걸리지 않았고, 바구니에 남은 게 있다면 마저 처리합니다.
if current_batch_texts and check_db_storage(db) < MAX_SAFE_STORAGE_MB:
    try:
        response = open_client.embeddings.create(
            input=current_batch_texts, 
            model=model,
            dimensions=reduced_dimensions
        )
        ops = [
            UpdateOne({"_id": _id}, {"$set": {"ingredients_embedding": data.embedding}})
            for _id, data in zip(current_batch_ids, response.data)
        ]
        collection.bulk_write(ops)
        processed_count += len(ops)
        print(f"🎉 최종 {processed_count}개 처리 완료! 남은 데이터까지 싹 비웠습니다.")
    except Exception as e:
        print(f"마지막 데이터 처리 중 에러: {e}")
elif current_batch_texts:
    print("마지막 데이터가 남았지만, 용량 초과 위험으로 처리를 생략했습니다.")

print("\n--- 모든 작업이 종료되었습니다 ---")

✅ 500개 처리 완료... (현재 DB 용량: 50.89MB)
✅ 1000개 처리 완료... (현재 DB 용량: 50.89MB)
✅ 1500개 처리 완료... (현재 DB 용량: 50.89MB)
✅ 2000개 처리 완료... (현재 DB 용량: 50.89MB)
✅ 2500개 처리 완료... (현재 DB 용량: 50.89MB)
✅ 3000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 3500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 4000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 4500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 5000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 5500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 6000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 6500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 7000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 7500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 8000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 8500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 9000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 9500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 10000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 10500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 11000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 11500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 12000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 12500개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 13000개 처리 완료... (현재 DB 용량: 82.44MB)
✅ 13500개 처리 완료... (현재 DB 용량: 82.

## 3. RAG 파이프라인. 함수 이름으로 하는 일 유추 가능.

### (1) 사용자 input에서 재료명만 추출하는 함수입니다.

In [123]:
def extract_ingredients_from_query(user_query):
    """사용자의 자연어 질문에서 식재료 키워드만 추출하여 쉼표로 연결합니다."""
    try:
        response = open_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "너는 요리 재료 추출기야. 사용자의 질문에서 '식재료' 명사만 정확히 추출해줘. "
                        "요리 이름이나 형용사, 동사는 제외해. "
                        "추출된 재료는 쉼표(,)로 구분해서 대답해. "
                        "예시 질문: '비 오는 날 먹기 좋은 따뜻한 어묵과 무 국물 요리' -> 대답: '어묵, 무'\n"
                        "만약 식재료가 없으면 '없음'이라고 대답해."
                    )
                },
                {"role": "user", "content": user_query}
            ],
            temperature=0 # 0으로 설정하여 창의성 없이 기계적으로 추출하게 함
        )
        extracted = response.choices[0].message.content.strip()
        print(f"🧠 [LLM 전처리] 추출된 재료 키워드: {extracted}")
        return extracted
    except Exception as e:
        print(f"⚠️ 재료 추출 오류: {e}")
        return user_query # 오류 시 원본 반환 (Fallback)

### (2) 사용자 input에서 추출한 재료명을 256차원의 벡터로 임베딩합니다.

In [124]:
def get_query_embedding(query_text):
    """사용자의 질문을 256차원 벡터로 변환합니다."""
    if not query_text or not query_text.strip():
        raise ValueError("입력된 질문이 비어있습니다.")
        
    try:
        # 🚨 주의: DB에 저장된 차원수(256)와 반드시 일치해야 합니다.
        response = open_client.embeddings.create(
            input=query_text,
            model="text-embedding-3-small",
            dimensions=256 
        )
        return response.data[0].embedding
    except RateLimitError:
        print("⚠️ [OpenAI API 오류] 요청 한도를 초과했습니다. 잠시 후 다시 시도합니다.")
        time.sleep(3)
        return None
    except APIError as e:
        print(f"⚠️ [OpenAI API 오류] 임베딩 생성 중 문제 발생: {e}")
        return None

### (3) 사용자 input의 embedding vector와 유사한 ingredients를 top3개 가져옵니다. 임계점은 0.75

In [125]:
def search_similar_recipes(query_vector, top_k=3, score_threshold=0.75):
    """MongoDB Atlas Vector Search를 통해 유사한 레시피를 찾습니다. 임계점은 0.75"""
    if not query_vector:
        return []

    # 🚨 중요: Atlas UI에서 'ingredients_embedding' 필드에 대한 
    # Vector Search Index(이름: vector_index)를 미리 만들어두어야 작동합니다.
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index", # 생성한 인덱스 이름
                "path": "ingredients_embedding",
                "queryVector": query_vector,
                "numCandidates": top_k * 10, # 검색 풀 크기 (보통 top_k의 10~20배)
                "limit": top_k
            }
        },
        {
            # 필요한 필드만 가져와 메모리 낭비 방지 및 속도 향상
            "$project": {
                "_id": 0,
                "title": 1,
                "url": 1,
                "ingredients": 1,
                "steps": 1,
                "score": {"$meta": "vectorSearchScore"} # 유사도 점수 (디버깅용)
            }
        }
    ]
    
    try:
        raw_results = list(collection.aggregate(pipeline))
        
        filtered_results = []
        for res in raw_results:
            if res.get("score", 0) >= score_threshold:
                filtered_results.append(res)
            
            if len(filtered_results) >= top_k:
                break
        
        return filtered_results
    except Exception as e:
        print(f"⚠️ [DB 검색 오류] 벡터 검색 중 문제가 발생했습니다: {e}")
        print("💡 팁: Atlas UI에서 Vector Search Index가 올바르게 생성되었는지 확인하세요.")
        return []

### (4) system 프롬프트를 구성합니다. vectordb에서 가져온 recipe들을 system prompt에 넣습니다(for문 +=).

In [133]:
def build_system_prompt(recipes):
    if not recipes:
        return "사용자의 요청에 맞는 레시피를 데이터베이스에서 찾지 못했습니다. 일반적인 요리 지식을 바탕으로 유연하게 대답해주세요."

    prompt = (
        "당신은 친절하고 전문적인 요리 추천 어시스턴트입니다.\n"
        "아래 제공된 [데이터베이스 검색 결과]를 최우선으로 참고하여 사용자의 질문에 답변해 주세요.\n\n"
        "### 데이터베이스 검색 결과 ###\n"
    )
    
    for idx, recipe in enumerate(recipes, 1):
        title = recipe.get("title", "제목 없음")
        url = recipe.get("url", "링크 없음") # 👈 url을 가져옵니다.
        ingredients = recipe.get("ingredients", [])
        steps = recipe.get("steps", [])
        
        cleaned_ingredients = [ing.replace(" 구매", "") for ing in ingredients]
        
        prompt += f"[{idx}] 요리명: {title}\n"
        prompt += f" - 링크: {url}\n"       # 👈 LLM이 읽을 수 있게 프롬프트에 추가합니다.
        prompt += f" - 재료: {', '.join(cleaned_ingredients)}\n"
        prompt += " - 조리 순서:\n"
        
        for step_num, step_text in enumerate(steps, 1):
            prompt += f"   {step_num}. {step_text}\n"
        prompt += "\n"
        
    prompt += (
        "### 지시사항 ###\n"
        "1. 위 레시피들을 활용하여 사용자가 만들 수 있는 요리를 친절하게 추천해주세요.\n"
        "2. (중요) 조리 순서나 URL 링크를 직접 출력할 필요는 없습니다. 그 정보는 시스템이 알아서 하단에 첨부할 것입니다.\n"
        "3. 각 요리의 특징이나 재료의 활용법 위주로 맛있게 요약 설명해 주세요.\n"
    )
    
    return prompt

### (5) 최종 프롬프트를 LLM에게 보냅니다.

In [127]:
def generate_final_answer(user_query, system_prompt):
    """최종 프롬프트를 LLM에 보내 답변을 생성합니다."""
    try:
        response = open_client.chat.completions.create(
            model="gpt-4o-mini", # 비용 효율적이고 빠른 최신 모델
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7 # 0.0(딱딱함) ~ 1.0(창의적) 사이의 값
            # max_tokens=800   # 🚨 너무 긴 답변이 나와 과금이 발생하는 것 방지
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"⚠️ [생성 오류] 답변을 생성하는 중 문제가 발생했습니다: {e}"

### (6) 모든 과정을 하나의 함수로 합칩니다.

In [136]:
def run_recipe_rag_pipeline(user_query):
    print(f"\n👤 사용자: {user_query}")
    print("-" * 50)
    
    # Step 1: 사용자 질문 임베딩
    print("1️⃣ 질문을 분석하는 중...")
    
    ingredient_keywords = extract_ingredients_from_query(user_query)
    query_vector = get_query_embedding(ingredient_keywords)


    if ingredient_keywords == "없음":
        return "질문에서 요리에 사용할 식재료를 찾지 못했습니다. 식재료를 포함해서 다시 질문해 주세요! (예: 감자랑 양파로 뭐 해 먹지?)"
            
    if not query_vector:
        return "죄송합니다. 현재 시스템에 일시적인 문제가 있어 질문을 분석할 수 없습니다."

 
    # Step 2: DB에서 유사 레시피 검색
    print("2️⃣ 적합한 레시피를 검색하는 중...")
    
    retrieved_recipes = search_similar_recipes(query_vector, top_k=3, score_threshold=0.75)
    if not retrieved_recipes:
        return f"현재 데이터베이스에 '{ingredient_keywords}' 재료 조합으로 추천할 만한 높은 매칭 점수의 레시피가 없습니다. 다른 재료로 검색해 보세요!"
   
    
    # Step 3: 프롬프트 조립
    print(f"3️⃣ {len(retrieved_recipes)}개의 레시피를 찾았습니다. 답변을 구성하는 중...")
    
    
    system_prompt = build_system_prompt(retrieved_recipes)
    
    # Step 4: LLM 답변 생성
    print("4️⃣ 최종 답변 생성 중...\n")
    final_answer = generate_final_answer(user_query, system_prompt)


    final_output = final_answer + "\n\n"
    final_output += "=" * 50 + "\n"
    final_output += "📌 [원본 레시피 상세 정보]\n"
    final_output += "=" * 50 + "\n\n"

    for idx, recipe in enumerate(retrieved_recipes, 1):
        title = recipe.get("title", "제목 없음")
        url = recipe.get("url", "링크 없음")
        steps = recipe.get("steps", [])
        score = recipe.get("score", 0) # 확인용 점수 출력 추가
        
        final_output += f"[{idx}] {title} (유사도: {score:.4f})\n"
        final_output += f"🔗 링크: {url}\n"
        final_output += "🍳 조리 순서:\n"
        
        # steps 배열을 그대로 반복해서 출력
        for step_num, step_text in enumerate(steps, 1):
            final_output += f"   {step_num}. {step_text}\n"
        
        final_output += "-" * 30 + "\n\n"
   
    return final_output

In [141]:
test_query = "어묵이랑 무로 먹을 수 있는 거"
result = run_recipe_rag_pipeline(test_query)
print("\n🤖 챗봇 답변:")
print(result)


👤 사용자: 어묵이랑 무로 먹을 수 있는 거
--------------------------------------------------
1️⃣ 질문을 분석하는 중...
🧠 [LLM 전처리] 추출된 재료 키워드: 어묵, 무
2️⃣ 적합한 레시피를 검색하는 중...
3️⃣ 3개의 레시피를 찾았습니다. 답변을 구성하는 중...
4️⃣ 최종 답변 생성 중...


🤖 챗봇 답변:
어묵과 무를 활용한 맛있는 요리로는 어묵탕이 있습니다. 어묵탕은 따뜻하고 국물이 진한 국물 요리로, 특히 찬바람이 부는 날에 먹기 좋습니다. 어묵을 다양한 종류로 준비할 수 있으며, 무를 함께 넣어 끓이면 국물이 더욱 시원하고 담백해집니다. 간장과 후추로 간을 맞추고, 마지막에 대파나 청양고추를 넣어 칼칼한 맛을 더하면 더욱 맛있게 즐길 수 있습니다.

또한, 에어프라이어를 이용해 어묵을 튀겨서 바삭하게 즐기는 방법도 있습니다. 얇게 자른 어묵을 에어프라이어에 넣고 바삭하게 구워주면 간편하게 안주로 즐길 수 있습니다. 두 가지 방법 모두 어묵과 무의 조화를 잘 살릴 수 있는 요리입니다. 

어떤 요리를 선택하시든, 어묵과 무의 조합이 정말 맛있을 거예요!

📌 [원본 레시피 상세 정보]

[1] 집밥TV) 찬바람 불땐 따끈한 어묵탕 맛있게 끓이기 (유사도: 0.8150)
🔗 링크: https://www.10000recipe.com/recipe/6945757
🍳 조리 순서:
   1. 어묵은 종류별로 먹기 좋게 썰어줍니다.
   2. 냄비에 물을 넣어 무를 넣고 끓입니다.
   3. 다른 냄비에 물을 끓여 어묵을 데쳐주세요. 데치면서 기름을 빼줍니다. 훨씬 담백해져요.
   4. 데친 어묵은 무를 넣고 끓인 냄비에 넣고 간장 2T를 넣어주세요.
   5. 소금으로 간을 합니다. 거의 다 익어가면 파와 고추를 넣어주세요. 기호에 따라 청양고추를 넣으면 더욱 칼칼해집니다.
------------------------------

[2] 초간단 어묵탕 끓이기 (유사도: 0.8147)
🔗 링크: h

## (참고) 트러블슈팅 했던 것들. 없어도 무방.

In [130]:
mismatch_count = collection.count_documents({
    "title": {"$regex": "국"},
    "category_name": "볶음"
})
print(f"데이터가 꼬인 레시피 수: {mismatch_count}개")

데이터가 꼬인 레시피 수: 11214개


In [79]:
def debug_vector_search(query_text):
    print(f"🔍 검색어: '{query_text}'")
    
    # 1. 벡터화
    query_vector = get_query_embedding(query_text)
    
    # 2. 파이프라인 (유사도 점수 포함)
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index", 
                "path": "ingredients_embedding",
                "queryVector": query_vector,
                "numCandidates": 100,
                "limit": 5
            }
        },
        {
            "$project": {
                "_id": 0, "title": 1, 
                "score": {"$meta": "vectorSearchScore"} # 👈 핵심! AI가 평가한 매칭 점수
            }
        }
    ]
    
    results = list(collection.aggregate(pipeline))
    
    print("-" * 40)
    for i, res in enumerate(results, 1):
        # 점수가 0.7 이상이면 꽤 괜찮은 매칭, 0.5 이하면 억지로 가져온 쓰레기값일 확률이 높습니다.
        print(f"[{i}위] {res.get('title')} (매칭 점수: {res.get('score'):.4f})")
    print("-" * 40)

# --- 2가지 케이스로 테스트해보세요 ---

# 테스트 1: 문장형 질문 (기존)
debug_vector_search("어묵과 무로 만들 수 있는 따뜻한 국물 요리 추천해줘")

# 테스트 2: 키워드형 질문 (해결책 검증)
debug_vector_search("어묵, 무")

🔍 검색어: '어묵과 무로 만들 수 있는 따뜻한 국물 요리 추천해줘'
----------------------------------------
[1위] 소고기 콩나물 파로밥 레시피 :) (매칭 점수: 0.7837)
[2위] 아이반찬 간장어묵볶음 레시피 (매칭 점수: 0.7808)
[3위] 두부김치 (매칭 점수: 0.7783)
[4위] 진라면 매운맛 맛있게 끓이는 법 (매칭 점수: 0.7769)
[5위] 김치어묵전골 만드는 법 김치 어묵 전골 레시피 어묵요리 어묵탕 캠핑요리 국물 요리 추천 (매칭 점수: 0.7761)
----------------------------------------
🔍 검색어: '어묵, 무'
----------------------------------------
[1위] 집밥TV) 찬바람 불땐 따끈한 어묵탕 맛있게 끓이기 (매칭 점수: 0.8150)
[2위] 초간단 어묵탕 끓이기 (매칭 점수: 0.8147)
[3위] 에어프라이어에게 어묵튀기기 (매칭 점수: 0.7982)
[4위] 어묵무국 (매칭 점수: 0.7887)
[5위] 어묵탕 (매칭 점수: 0.7821)
----------------------------------------


## (참고) GPT 추천 프로젝트 구조

### 1) app/utils/config.py - 환경변수를 로드하고 OpenAI 클라이언트를 한 곳에서 초기화하여 다른 파일에서 재사용합니다.

### 2) app/db/mongo_db.py - MongoDB 연결 로직을 담당합니다.

### 3) app/rag/prompts.py - 프롬프트 내용이 길어지면 메인 로직이 지저분해지므로, 문자열과 프롬프트 생성 로직만 따로 뺍니다.

### 4) app/rag/retriever.py - 임베딩 변환 및 MongoDB Vector Search 로직을 담당합니다.

### 5) app/rag/pipeline.py - LLM 호출(추출, 최종 답변 생성) 및 위의 모든 모듈을 조합하여 실제 RAG 파이프라인을 실행합니다.

### 6) 이렇게 모듈화를 마치면, 나중에 app/api/main.py나 frontend/app.py에서 파이프라인을 호출할 때 단 한 줄이면 충분합니다.